# 🔮 Evaluasi Model Prophet (Backtesting)
---

Notebook ini dibuat untuk melakukan penelitian pribadi (research) mengenai performa model Meta Prophet dalam memprediksi harga komoditas historis di Aceh.

Kita akan menggunakan teknik **Train-Test Split (Holdout)** dengan menyembunyikan 90 hari terakhir dari data historis sebagai *Data Testing*.

### 1. Load & Bersihkan Data
*(Cell di bawah ini sudah digabungkan dengan Library Import agar tidak terjadi error eksekusi)*

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from prophet import Prophet
import matplotlib.pyplot as plt
import logging
import os

# Mematikan pesan log dari Prophet agar output lebih rapi
logging.getLogger('prophet').setLevel(logging.ERROR)
logging.getLogger('cmdstanpy').setLevel(logging.ERROR)

# Set base directoy ke root proyek secara absolut agar tidak ada error path
current_dir = Path(os.getcwd())
if current_dir.name == 'scripts':
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir
    
DATA_DIR = PROJECT_ROOT / 'data'

def load_and_clean_excel(filepath, year):
    raw = pd.read_excel(filepath, header=None)
    date_strings = raw.iloc[0, 2:].values
    dates = pd.to_datetime(date_strings, format='%d/ %m/ %Y')
    roman_numerals = {'I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX', 'X'}
    data_rows = raw.iloc[1:]
    mask = ~data_rows.iloc[:, 0].isin(roman_numerals)
    commodity_data = data_rows[mask].copy()
    records = []
    for _, row in commodity_data.iterrows():
        commodity_name = row.iloc[1].strip()
        prices = row.iloc[2:].values
        for date, price in zip(dates, prices):
            price_str = str(price).strip()
            if pd.isna(price) or price_str in ('', '-', 'nan'):
                price_val = np.nan
            else:
                price_val = float(price_str.replace(',', ''))
            records.append({
                'date': date,
                'commodity': commodity_name,
                'price': price_val,
                'year': year
            })
    return pd.DataFrame(records)

print("Loading data...")
dfs = []
for y in [2023, 2024, 2025]:
    file_path = DATA_DIR / f'{y}.xlsx'
    if file_path.exists():
        dfs.append(load_and_clean_excel(file_path, y))
    else:
        print(f"Warning: {file_path} tidak ditemukan.")

if len(dfs) > 0:
    df = pd.concat(dfs, ignore_index=True)
    df['date'] = pd.to_datetime(df['date'])
    df_clean = df.dropna(subset=['price']).copy()
    commodities = sorted(df_clean['commodity'].unique())
    print(f"✅ Berhasil memuat {len(df_clean)} baris data.")
else:
    print("❌ Gagal memuat data! Dataset Excel tidak ditemukan sama sekali di ", DATA_DIR)

### 2. Tentukan Data Training dan Data Testing

In [ ]:
# Kita ambil 90 hari terakhir sebagai periode Test
latest_date = df_clean['date'].max()
test_start_date = latest_date - pd.Timedelta(days=90)

print(f"Tanggal akhir data   : {latest_date.strftime('%Y-%m-%d')}")
print(f"Batas Train/Test     : {test_start_date.strftime('%Y-%m-%d')}")
print("\n--- Rencana Ujian ---")
print(f"Data Training : Awal s/d {test_start_date.strftime('%Y-%m-%d')} (Model belajar dari sini)")
print(f"Data Testing  : {test_start_date.strftime('%Y-%m-%d')} s/d {latest_date.strftime('%Y-%m-%d')} (Kita tes kemampuan model di sini)")

### 3. Eksekusi Evaluasi Model (Looping 18 Komoditas)
Di sini kita akan mencatat MAE, RMSE, dan MAPE.

In [ ]:
results = []
predictions_store = {}

for commodity in commodities:
    # Siapkan DF format Prophet (ds = date, y = price)
    cdf = df_clean[df_clean['commodity'] == commodity].sort_values('date').copy()
    pdf = cdf[['date', 'price']].rename(columns={'date': 'ds', 'price': 'y'})
    
    # Split Data
    train = pdf[pdf['ds'] < test_start_date].copy()
    test = pdf[pdf['ds'] >= test_start_date].copy()
    
    if len(test) == 0:
        continue
        
    try:
        # Inisialisasi Model
        model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        model.fit(train)
        
        # Prediksi khusus untuk rentang tanggal Test
        future = test[['ds']].copy()
        forecast = model.predict(future)
        
        # Gabungkan Prediksi vs Aktual
        evaluation_df = pd.merge(test, forecast[['ds', 'yhat']], on='ds')
        predictions_store[commodity] = evaluation_df
        
        # Hitung Metrik
        y_true = evaluation_df['y']
        y_pred = evaluation_df['yhat']
        
        mae = np.mean(np.abs(y_true - y_pred))
        rmse = np.sqrt(np.mean((y_true - y_pred)**2))
        mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
        
        results.append({
            'Commodity': commodity,
            'MAE (Rp)': round(mae, 0),
            'RMSE (Rp)': round(rmse, 0),
            'MAPE (%)': round(mape, 2)
        })
    except Exception as e:
        print(f"⚠️ Gagal memproses {commodity}: {e}")

# Tampilkan DataFrame Hasil
res_df = pd.DataFrame(results).sort_values('MAPE (%)').reset_index(drop=True)
res_df

### 4. Visualisasi untuk Semua Komoditas
Cell di bawah ini akan secara otomatis melakukan perulangan (*loop*) dan mencetak grafik Aktul vs Prediksi untuk **seluruh ke-18 komoditas** sekaligus.

In [ ]:
for commodity in res_df['Commodity'].values: # Looping berdasarkan urutan tingkat Error terkecil (diurutkan dari res_df)
    if commodity in predictions_store:
        eval_df = predictions_store[commodity]
        
        plt.figure(figsize=(10, 4)) # Ukuran sedikit diperkecil agar pas saat di-scroll
        plt.plot(eval_df['ds'], eval_df['y'], label='Harga Asli (Actual)', linestyle='-', marker='o', markersize=3, color='#3b82f6')
        plt.plot(eval_df['ds'], eval_df['yhat'], label='Tebakan AI (Predicted)', linestyle='--', color='#ef4444')
        
        # Dapatkan Metrik untuk dicetak di judul
        metric_row = res_df[res_df['Commodity'] == commodity].iloc[0]
        mape_val = metric_row['MAPE (%)']
        
        plt.title(f'{commodity} - MAE: Rp {metric_row["MAE (Rp)"]:.0f} | Error: {mape_val}%', fontweight='bold')
        plt.ylabel('Rupiah')
        plt.legend(loc='best')
        plt.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()